# DSCD Plots and Scaling Figures

This notebook regenerates DSCD paper figures from `output-dsfb-dscd/<timestamp>/` exports.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Parameter cell
OUTPUT_DIR = None  # e.g. '/content/dsfb/output-dsfb-dscd/20260303_120000'
RUN_QUICK_DEMO = True
AUTO_INSTALL_RUST = True
AUTO_CLONE_REPO = True
REPO_URL = 'https://github.com/infinityabundance/dsfb.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/dsfb')
OUTPUT_ROOT_NAME = 'output-dsfb-dscd'

def ensure_cargo_available(auto_install: bool = True):
    if shutil.which('cargo'):
        return
    cargo_bin = Path.home() / '.cargo' / 'bin'
    if cargo_bin.exists():
        os.environ['PATH'] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if shutil.which('cargo'):
        return
    if not auto_install:
        raise FileNotFoundError("'cargo' is not in PATH. Set RUN_QUICK_DEMO=False or install Rust.")
    if 'google.colab' not in sys.modules:
        raise FileNotFoundError("'cargo' is not in PATH. Install Rust, or set RUN_QUICK_DEMO=False.")
    print('Installing Rust toolchain (cargo) for Colab...')
    subprocess.run(
        ['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal'],
        check=True,
    )
    os.environ['PATH'] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if not shutil.which('cargo'):
        raise FileNotFoundError("Rust installed but 'cargo' is still missing from PATH.")

def find_workspace_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        cargo_toml = candidate / 'Cargo.toml'
        if not cargo_toml.exists():
            continue
        text = cargo_toml.read_text(encoding='utf-8', errors='ignore')
        if 'dsfb-dscd' in text:
            return candidate
    return None

def ensure_workspace_root(auto_clone: bool = True) -> Path:
    root = find_workspace_root(Path.cwd().resolve())
    if root is not None:
        return root
    if not auto_clone:
        raise FileNotFoundError(
            'Workspace Cargo.toml not found. Run this notebook from repo root or set AUTO_CLONE_REPO=True.'
        )
    if not REPO_DIR.exists():
        print(f'Cloning repo into {REPO_DIR} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    root = find_workspace_root(REPO_DIR.resolve())
    if root is None:
        raise FileNotFoundError(f'Failed to locate workspace after cloning {REPO_URL}')
    return root

def run_checked(cmd: list[str], cwd: Path):
    print('Running:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end='' if proc.stdout.endswith('\n') else '\n')
    if proc.stderr:
        print(proc.stderr, end='' if proc.stderr.endswith('\n') else '\n', file=sys.stderr)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)

def run_dscd_demo(workspace_root: Path):
    cmd_mode = ['cargo', 'run', '-p', 'dsfb-dscd', '--release', '--', '--mode', 'demo']
    try:
        run_checked(cmd_mode, cwd=workspace_root)
        return
    except subprocess.CalledProcessError as exc:
        stderr = exc.stderr or ''
        if "unexpected argument '--mode'" not in stderr:
            raise
        print('Detected legacy DSCD CLI, retrying with --quick ...')
    cmd_quick = ['cargo', 'run', '-p', 'dsfb-dscd', '--release', '--', '--quick']
    run_checked(cmd_quick, cwd=workspace_root)

WORKSPACE_ROOT = ensure_workspace_root(auto_clone=AUTO_CLONE_REPO)
OUTPUT_ROOT = WORKSPACE_ROOT / OUTPUT_ROOT_NAME
print('Using WORKSPACE_ROOT:', WORKSPACE_ROOT)

if RUN_QUICK_DEMO:
    ensure_cargo_available(auto_install=AUTO_INSTALL_RUST)
    run_dscd_demo(WORKSPACE_ROOT)

if OUTPUT_DIR is None:
    runs = sorted([p for p in OUTPUT_ROOT.glob('*') if p.is_dir()])
    if not runs:
        raise FileNotFoundError(f'No DSCD outputs found under {OUTPUT_ROOT}. Set OUTPUT_DIR or keep RUN_QUICK_DEMO=True.')
    OUTPUT_DIR = str(runs[-1])

OUTPUT_DIR = str(Path(OUTPUT_DIR).resolve())
print('Using OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

RUN_DIR = Path(OUTPUT_DIR)

def save_fig(name: str):
    path = RUN_DIR / name
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print('saved', path)

summary_path = RUN_DIR / 'threshold_scaling_summary.csv'
summary = pd.read_csv(summary_path)
curve_paths = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))
if not curve_paths:
    raise FileNotFoundError('No threshold_curve_N_<N>.csv files found in run dir')

curve_data = []
for p in curve_paths:
    n = int(p.stem.split('_')[-1])
    df = pd.read_csv(p)
    df['num_events'] = n
    curve_data.append(df)

curves = pd.concat(curve_data, ignore_index=True)
available_ns = sorted(curves['num_events'].unique())
print('Loaded N values:', available_ns)

In [ ]:
# (A) Expansion curves across N
plt.figure(figsize=(8, 5))
for n in available_ns:
    df = curves[curves['num_events'] == n]
    plt.plot(df['tau'], df['expansion_ratio'], linewidth=2, label=f'N={n}')
plt.xlabel('tau')
plt.ylabel('expansion_ratio')
plt.title('DSCD Expansion Curves vs Tau')
plt.grid(alpha=0.25)
plt.legend()
save_fig('fig_dscd_expansion_curves_vs_tau.png')
plt.show()

In [ ]:
# (B) Finite-size scaling plots
summary = summary.sort_values('num_events')
N = summary['num_events'].to_numpy(dtype=float)
width = summary['transition_width'].to_numpy(dtype=float)
max_d = summary['max_derivative'].to_numpy(dtype=float)

valid_width = (N > 0) & (width > 0)
if valid_width.sum() >= 2:
    x = np.log(N[valid_width])
    y = np.log(width[valid_width])
    slope, intercept = np.polyfit(x, y, 1)
    y_hat = slope * x + intercept
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot if ss_tot > 0 else 0.0)
    gamma = -slope
    print(f'width ~ N^(-gamma): gamma={gamma:.4f}, R^2={r2:.4f}')
else:
    print('Not enough positive width points to fit gamma.')

plt.figure(figsize=(7, 4.5))
plt.loglog(N, np.maximum(width, 1e-12), marker='o', linewidth=2)
plt.xlabel('N')
plt.ylabel('transition_width')
plt.title('Transition Width vs N')
plt.grid(alpha=0.25, which='both')
save_fig('fig_dscd_transition_width_vs_N.png')
plt.show()

plt.figure(figsize=(7, 4.5))
plt.loglog(N, np.maximum(max_d, 1e-12), marker='o', linewidth=2)
plt.xlabel('N')
plt.ylabel('max_derivative')
plt.title('Max Derivative vs N')
plt.grid(alpha=0.25, which='both')
save_fig('fig_dscd_max_derivative_vs_N.png')
plt.show()

In [ ]:
# (C) Single-N step plot (hero)
largest_n = max(available_ns)
hero_df = curves[curves['num_events'] == largest_n]
plt.figure(figsize=(8, 5))
plt.plot(hero_df['tau'], hero_df['expansion_ratio'], linewidth=2.5)
plt.xlabel('tau')
plt.ylabel('expansion_ratio')
plt.title(f'DSCD Expansion Ratio vs Tau (N={largest_n})')
plt.grid(alpha=0.25)
save_fig('fig_dscd_expansion_ratio_vs_tau.png')
plt.show()

In [ ]:
# (D) Degree distributions
degree_df = pd.read_csv(RUN_DIR / 'degree_distribution.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degree_df['in_degree'], bins=40)
axes[0].set_title('In-degree distribution')
axes[0].set_xlabel('in_degree')
axes[0].set_ylabel('count')
axes[1].hist(degree_df['out_degree'], bins=40)
axes[1].set_title('Out-degree distribution')
axes[1].set_xlabel('out_degree')
axes[1].set_ylabel('count')
for ax in axes:
    ax.grid(alpha=0.25)
save_fig('fig_dscd_degree_distribution.png')
plt.show()

In [ ]:
# (E) Interval size distribution
interval_df = pd.read_csv(RUN_DIR / 'interval_sizes.csv')
plt.figure(figsize=(8, 5))
plt.hist(interval_df['interval_size'], bins=50)
plt.xlabel('interval_size')
plt.ylabel('count')
plt.title('Causal Interval Size Distribution')
plt.grid(alpha=0.25)
save_fig('fig_dscd_interval_sizes.png')
plt.show()

In [ ]:
# (F) Path length distribution
path_df = pd.read_csv(RUN_DIR / 'path_lengths.csv')
plt.figure(figsize=(8, 5))
plt.hist(path_df['longest_path_length'], bins=50)
plt.xlabel('longest_path_length')
plt.ylabel('count')
plt.title('Longest Path Length Distribution')
plt.grid(alpha=0.25)
save_fig('fig_dscd_path_length_distribution.png')
plt.show()

In [ ]:
# (G) Graph preview
edges_df = pd.read_csv(RUN_DIR / 'graph_edges.csv')
preview_edges = edges_df.head(512)
G_preview = nx.DiGraph()
for _, row in preview_edges.iterrows():
    G_preview.add_edge(int(row['src']), int(row['dst']))

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_preview, seed=42)
nx.draw_networkx_edges(G_preview, pos, arrows=True, width=0.8, alpha=0.6)
nx.draw_networkx_nodes(G_preview, pos, node_size=18)
plt.title('DSCD Graph Preview (first 512 edges)')
plt.axis('off')
save_fig('fig_dscd_graph_preview.png')
plt.show()

In [ ]:
# (H) Edge provenance panel
events_df = pd.read_csv(RUN_DIR / 'graph_events.csv')
prov_df = pd.read_csv(RUN_DIR / 'edge_provenance.csv')
if prov_df.empty:
    raise ValueError('edge_provenance.csv is empty')
row = prov_df.iloc[0]
src = int(row['src'])
dst = int(row['dst'])

G_full = nx.DiGraph()
for _, e in edges_df.iterrows():
    G_full.add_edge(int(e['src']), int(e['dst']))

nodes = {src, dst}
nodes.update(nx.single_source_shortest_path_length(G_full, src, cutoff=1).keys())
nodes.update(nx.single_source_shortest_path_length(G_full.reverse(copy=False), src, cutoff=1).keys())
nodes.update(nx.single_source_shortest_path_length(G_full, dst, cutoff=1).keys())
nodes.update(nx.single_source_shortest_path_length(G_full.reverse(copy=False), dst, cutoff=1).keys())
H = G_full.subgraph(nodes).copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pos = nx.spring_layout(H, seed=7)
nx.draw_networkx_edges(H, pos, ax=axes[0], arrows=True, width=1.0, alpha=0.7)
node_colors = ['tab:red' if n in {src, dst} else 'tab:blue' for n in H.nodes()]
node_sizes = [180 if n in {src, dst} else 60 for n in H.nodes()]
nx.draw_networkx_nodes(H, pos, ax=axes[0], node_color=node_colors, node_size=node_sizes)
axes[0].set_title('Local neighborhood around selected edge')
axes[0].axis('off')

src_event = events_df[events_df['id'] == src].iloc[0]
dst_event = events_df[events_df['id'] == dst].iloc[0]
text = '\n'.join([
    f'src: {src}',
    f'dst: {dst}',
    f'observer/module_id: {int(row["module_id"])}',
    f'trust_at_creation: {row["trust_at_creation"]:.6f}',
    f'rewrite_rule_id: {row["rewrite_rule_id"]}',
    f'residual_state_src: {src_event["residual_state"]}',
    f'residual_state_dst: {dst_event["residual_state"]}',
])
axes[1].axis('off')
axes[1].text(0.02, 0.98, text, va='top', family='monospace')
axes[1].set_title('Edge provenance summary')
save_fig('fig_dscd_edge_provenance.png')
plt.show()

In [ ]:
# (I) Regime-change overlay from DSCD test run outputs
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
curve_paths = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))
if curve_paths:
    curve_path = curve_paths[-1]
    n_val = int(curve_path.stem.split('_')[-1])
    curve_df = pd.read_csv(curve_path).sort_values('tau')
    t = pd.to_numeric(curve_df['tau'], errors='coerce')
    residual_level = pd.to_numeric(curve_df['expansion_ratio'], errors='coerce')
    valid = t.notna() & residual_level.notna()
    t = t[valid].to_numpy()
    residual_level = residual_level[valid].to_numpy()

    tau_star = None
    if 'tau_star' in summary.columns and 'num_events' in summary.columns:
        row = summary[summary['num_events'] == n_val]
        if not row.empty:
            tau_star = float(row.iloc[0]['tau_star'])
    if tau_star is None and t.size > 0:
        tau_star = float(t[t.size // 2])
    regime_flag = (t >= tau_star).astype(float) if tau_star is not None else np.zeros_like(t)

    axes[0].plot(t, residual_level, linewidth=1.8, label=f'expansion_ratio (N={n_val})')
    axes[0].plot(t, regime_flag, linewidth=1.2, label='regime_flag (tau >= tau*)')
    axes[0].set_ylabel('DSCD observable')
    axes[0].set_title('DSCD regime-change overlay (from test run)')
    axes[0].legend(loc='best')

    if 'tau_star' in summary.columns:
        ratio_proxy = summary['tau_star']
        proxy_label = 'tau_star'
    else:
        ratio_proxy = summary['transition_width']
        proxy_label = 'transition_width'
    axes[1].plot(summary['num_events'], ratio_proxy, marker='o')
    axes[1].set_ylabel(f'DSCD proxy ({proxy_label})')
    axes[1].set_xlabel('N')
else:
    axes[0].text(0.02, 0.5, 'No threshold_curve_N_<N>.csv found in DSCD run output.', transform=axes[0].transAxes)
    axes[1].text(0.02, 0.5, 'Run DSCD quick demo to generate threshold curves.', transform=axes[1].transAxes)
    axes[0].set_ylabel('DSCD observable')
    axes[1].set_ylabel('DSCD proxy')
    axes[1].set_xlabel('N')

for ax in axes:
    ax.grid(alpha=0.25)
save_fig('fig_dscd_plasma_regime_overlay.png')
plt.show()

In [ ]:
# (J) Run-to-run determinism check
root = Path('output-dsfb-dscd')
runs = sorted([p for p in root.glob('*') if p.is_dir()])
fig, ax = plt.subplots(figsize=(8, 5))
if len(runs) >= 2:
    r1, r2 = runs[-2], runs[-1]
    c1 = sorted(r1.glob('threshold_curve_N_*.csv'))
    c2 = sorted(r2.glob('threshold_curve_N_*.csv'))
    common = sorted(set(p.name for p in c1) & set(p.name for p in c2))
    if common:
        target = common[-1]
        d1 = pd.read_csv(r1 / target)
        d2 = pd.read_csv(r2 / target)
        ax.plot(d1['tau'], d1['expansion_ratio'], label=f'{r1.name}/{target}', linewidth=2)
        ax.plot(d2['tau'], d2['expansion_ratio'], '--', label=f'{r2.name}/{target}', linewidth=2)
        ax.set_title('Replay consistency overlay')
        ax.set_xlabel('tau')
        ax.set_ylabel('expansion_ratio')
        ax.legend()
    else:
        ax.text(0.02, 0.5, 'No matching threshold_curve_N_<N>.csv in the last two runs.', transform=ax.transAxes)
else:
    ax.text(0.02, 0.5, 'Need at least two timestamped runs for determinism check.\nRun quick mode twice with same config.', transform=ax.transAxes)
ax.grid(alpha=0.25)
save_fig('fig_dscd_replay_consistency.png')
plt.show()